In [ ]:
!nvidia-smi

In [ ]:
import torch
import sys

print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU count:", torch.cuda.device_count())
    print(
        "VRAM:",
        round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2),
        "GB"
    )

In [ ]:
%cd /kaggle/working

!git clone https://github.com/Taipan-Enigma/DiffRhythm-Fine-tuner.git

In [ ]:
%cd /kaggle/working/DiffRhythm-Fine-tuner
!ls

In [ ]:
!apt-get update -qq
!apt-get install -y espeak-ng ffmpeg libsndfile1

In [ ]:
%pip install torch==2.6.0 torchaudio==2.6.0

In [ ]:
%pip install -r requirements.txt

In [ ]:
import torch
import torchaudio
import transformers
import muq
import phonemizer
import torchdiffeq

print("All imports successful")
print("CUDA:", torch.cuda.is_available())

In [ ]:
%cd /kaggle/working/DiffRhythm-Fine-tuner

!ls -lh infer/example/

In [ ]:
!cat infer/example/eg_en_full.lrc

In [ ]:
%cd /kaggle/working/DiffRhythm-Fine-tuner

!mkdir -p g2p/sources/g2p_chinese_model

!wget -O g2p/sources/g2p_chinese_model/poly_bert_model.onnx \
https://huggingface.co/spaces/ASLP-lab/DiffRhythm/resolve/main/diffrhythm/g2p/sources/g2p_chinese_model/poly_bert_model.onnx

In [ ]:
!ls -lh g2p/sources/g2p_chinese_model/

In [ ]:
%cd /kaggle/working/DiffRhythm-Fine-tuner

from huggingface_hub import snapshot_download
from pathlib import Path
import shutil

download_path = snapshot_download(
    repo_id="ASLP-lab/DiffRhythm",
    repo_type="space",
    allow_patterns=["diffrhythm/g2p/sources/*"]
)

source = Path(download_path) / "diffrhythm/g2p/sources"
destination = Path("g2p/sources")

shutil.copytree(source, destination, dirs_exist_ok=True)

print("G2P resources downloaded successfully")

In [ ]:
!ls -lh g2p/sources/
!ls -lh g2p/sources/g2p_chinese_model/

In [ ]:
%pip install --no-deps py3langid==0.3.0

In [ ]:
!python -c "from py3langid.langid import LanguageIdentifier; print(hasattr(LanguageIdentifier, 'from_pickled_model'))"

In [ ]:
!PYTHONPATH=. python -c "from infer.infer_utils import CNENTokenizer; tokenizer = CNENTokenizer(); print('Tokenizer initialized successfully!')"

In [ ]:
!wget -O infer/example/eg_en.mp3 \
https://raw.githubusercontent.com/ASLP-lab/DiffRhythm/main/infer/example/eg_en.mp3

In [ ]:
!ls -lh infer/example/eg_en.mp3

In [ ]:
!PYTHONPATH=. python infer/infer.py \
    --lrc-path infer/example/eg_en_full.lrc \
    --ref-audio-path infer/example/eg_en.mp3 \
    --audio-length 177 \
    --output-dir infer/example/output \
    --chunked \
    --batch-infer-num 1

In [ ]:
%pip install datasets soundfile

In [ ]:
from datasets import load_dataset, Audio

ds = load_dataset(
    "jamendolyrics/jamendolyrics",
    "en",
    split="test",
    streaming=True
)

ds = ds.cast_column("audio", Audio(decode=False))

In [ ]:
%pip install --no-deps datasets==3.6.0

In [ ]:
import datasets
import torch

print("Datasets:", datasets.__version__)
print("PyTorch:", torch.__version__)

In [2]:
from datasets import load_dataset, Audio

ds = load_dataset(
    "jamendolyrics/jamendolyrics",
    "en",
    split="test",
    streaming=True
)

ds = ds.cast_column("audio", Audio(decode=False))

Resolving data files:   0%|          | 0/80 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/21 [00:00<?, ?it/s]

In [3]:
song = next(
    s for s in ds
    if not s["lyric_overlap"]
)

print("Title:", song["title"])
print("Artist:", song["artist"])
print("Genre:", song["genre"])
print("First lyric line:", song["lines"][0])

Title: Give Me The Same
Artist: HILA
Genre: Pop
First lyric line: {'start': 18.6199798584, 'end': 19.8730163574, 'text': 'lay awake at night'}


In [5]:
from pathlib import Path
from huggingface_hub import hf_hub_download
import shutil

folder = Path("experiments/jamendo_001")
folder.mkdir(parents=True, exist_ok=True)

downloaded_audio = hf_hub_download(
    repo_id="jamendolyrics/jamendolyrics",
    repo_type="dataset",
    filename="subsets/en/mp3/HILA_-_Give_Me_the_Same.mp3",
    revision="de188c963fd4539bc769b3feb83582e5a9595e36"
)

audio_path = folder / "original.mp3"

shutil.copyfile(downloaded_audio, audio_path)

print("Title:", song["title"])
print("Artist:", song["artist"])
print("Saved:", audio_path)

subsets/en/mp3/HILA_-_Give_Me_the_Same.m(…):   0%|          | 0.00/5.52M [00:00<?, ?B/s]

Title: Give Me The Same
Artist: HILA
Saved: experiments/jamendo_001/original.mp3


In [6]:
print("Number of lines:", len(song["lines"]))
print(song["lines"][:3])

Number of lines: 58
[{'start': 18.6199798584, 'end': 19.8730163574, 'text': 'lay awake at night'}, {'start': 20.8442592621, 'end': 22.1282539368, 'text': 'wondering how could i'}, {'start': 23.0229873657, 'end': 24.4241275787, 'text': 'let it get this way'}]


In [7]:
def to_lrc_time(seconds):
    cs = round(seconds * 100)
    minutes, remainder = divmod(cs, 6000)
    seconds, centiseconds = divmod(remainder, 100)

    return f"[{minutes:02d}:{seconds:02d}.{centiseconds:02d}]"

duration = 95

selected_lines = [
    line for line in song["lines"]
    if 0 <= line["start"] and line["end"] < 94
]

assert selected_lines, "No valid lyric lines found"

lrc = "\n".join(
    to_lrc_time(line["start"]) + line["text"].strip()
    for line in selected_lines
)

lrc += "\n[01:34.00]End\n"

lrc_path = folder / "lyrics.lrc"
lrc_path.write_text(lrc, encoding="utf-8")

print(lrc)

[00:18.62]lay awake at night
[00:20.84]wondering how could i
[00:23.02]let it get this way
[00:27.46]through all the pain
[00:29.48]i took all the blame
[00:31.94]while you cursed my last name
[00:36.19]i thought we could get better
[00:38.45]stayed committed like a soldier
[00:40.97]believed we're okay
[00:44.42]now i know that you don't care at all
[00:53.57]time wasted
[00:55.24]gave all of my soul and my heart to someone that would never
[01:01.11]give me the same
[01:02.52]i was a fool to believe that you would finally be the one to
[01:09.95]give me the same
[01:11.29]heartbroken now
[01:15.79]how you let me down
[01:19.64]but somehow deep down i still do
[01:26.00]love you
[01:31.89]now your out all night
[01:34.00]End



In [8]:
!ls -lh experiments/jamendo_001/

total 5.3M
-rw-r--r-- 1 root root  736 Nov 22 10:06 lyrics.lrc
-rw-r--r-- 1 root root 5.3M Nov 22 10:06 original.mp3


In [9]:
!PYTHONPATH=. python infer/infer.py \
    --lrc-path experiments/jamendo_001/lyrics.lrc \
    --ref-audio-path experiments/jamendo_001/original.mp3 \
    --audio-length 95 \
    --output-dir experiments/jamendo_001/generated \
    --chunked \
    --batch-infer-num 1

Current working directory: /kaggle/working/DiffRhythm-Fine-tuner
cfm_model.pt: 100%|█████████████████████████| 2.22G/2.22G [00:05<00:00, 400MB/s]
/usr/local/lib/python3.13/dist-packages/jieba/_compat.py:18: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
./g2p/sources/g2p_chinese_model/poly_bert_model.onnx
/usr/local/lib/python3.13/dist-packages/onnxruntime/capi/onnxruntime_inference_collection.py:197: UserWarning: Specified provider 'CUDAExecutionProvider' is not in available provider names.Available providers: 'AzureExecutionProvider, CPUExecutionProvider'
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/torch/nn/utils/weight_norm.py:143: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightN

In [10]:
from IPython.display import FileLink, display

print("Original song:")
display(FileLink("experiments/jamendo_001/original.mp3"))

print("Generated song:")
display(FileLink("experiments/jamendo_001/generated/output.wav"))

Original song:


/kaggle/working/DiffRhythm-Fine-tuner/experiments/jamendo_001/original.mp3

Generated song:


/kaggle/working/DiffRhythm-Fine-tuner/experiments/jamendo_001/generated/output.wav

In [11]:
import shutil

shutil.make_archive(
    "/kaggle/working/jamendo_comparison",
    "zip",
    "experiments/jamendo_001"
)

'/kaggle/working/jamendo_comparison.zip'

In [12]:
%cd /kaggle/working

display(FileLink("jamendo_comparison.zip"))

/kaggle/working


/kaggle/working/jamendo_comparison.zip